# Optical Flow

### What Is Optical Flow?

YOLO asks:
- what object is in this region?

Optical flow asks:
- where did this pixel move between frame 1 and frame 2?

No classes, no bounding boxes, no labels:
- just raw pixel movement
- no neural network needed
- works on any moving object
- tells us direction and speed of movement

Example — a dot moving right:

```
Frame 1:        Frame 2:        Optical Flow:
. . . . .       . . . . .       . . → . .
. . o . .  →   . . . o .  →   . . . . .
. . . . .       . . . . .       . . . . .
```

### Dense vs Sparse

**Dense optical flow:**
- calculates movement for every single pixel
- full motion picture of entire scene
- computationally heavy

**Sparse optical flow:**
- tracks only specific interesting points
- pick ~100 good corners, track only those
- much faster

We use sparse — a few key points tell us everything about movement without processing every pixel.

### Good Features To Track

Not all pixels are equally trackable.

Bad point — flat blue sky:
- every surrounding pixel looks identical
- if it moves 5 pixels → still looks exactly the same
- impossible to confirm it moved

Good point — chess board corner:
- distinct color change in 2 directions
- if it moves 5 pixels → clearly in a different place
- easy to confirm movement

This uniqueness is called **cornerness**.

`cv2.goodFeaturesToTrack()` finds corners with a quality threshold:
- minimum quality score
- minimum distance between points (no clustering)
- strong enough contrast to survive across frames

A regular corner detector finds any corner.
`goodFeaturesToTrack` only returns corners worth tracking.

### Lucas-Kanade Algorithm

The most widely used sparse optical flow method:
- takes previous frame, current frame, and points to track
- returns new positions of those points
- returns a status flag for each point (tracked successfully or lost)

Pipeline:

```
Frame 1
↓
goodFeaturesToTrack() → find ~100 good corners
↓
Frame 2
↓
calcOpticalFlowPyrLK() → where did those corners move?
↓
Draw arrows showing movement
```

#### Why PyrLK?

`Pyr` = pyramid:
- runs on multiple downscaled versions of the frame
- coarse level catches large movements
- fine level refines small movements
- handles both fast and slow motion reliably

### Setup

In [ ]:
import cv2
import numpy as np

### Constants

In [ ]:
# goodFeaturesToTrack parameters
MAX_CORNERS    = 100     # maximum points to track
QUALITY        = 0.3     # minimum quality score (0-1)
MIN_DISTANCE   = 7       # minimum pixels between points

# Lucas-Kanade parameters
LK_PARAMS = dict(
    winSize  = (15, 15),
    maxLevel = 2,
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
)

# Minimum movement to draw an arrow
MIN_MOVEMENT = 2

#### `MAX_CORNERS`
- maximum number of points to track per frame
- more points = more coverage, more CPU

#### `QUALITY`
- minimum cornerness score to be selected
- higher = fewer but more reliable points

#### `MIN_DISTANCE`
- prevents multiple points clustering on the same corner
- ensures spread across the whole frame

#### `winSize`
- size of search window around each point
- larger = handles bigger movements, slower

#### `maxLevel`
- number of pyramid levels
- 2 = runs on 3 scales (original, half, quarter)

#### `criteria`
- when to stop refining the point position
- stops after 10 iterations OR when movement < 0.03 pixels

#### `MIN_MOVEMENT`
- filters out points that barely moved
- prevents noise from being drawn as arrows

### Webcam Loop

In [ ]:
cap = cv2.VideoCapture(0)

ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

prev_points = cv2.goodFeaturesToTrack(
    prev_gray,
    maxCorners=MAX_CORNERS,
    qualityLevel=QUALITY,
    minDistance=MIN_DISTANCE
)

while True:

    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Track points
    if prev_points is not None and len(prev_points) > 0:

        next_points, status, error = cv2.calcOpticalFlowPyrLK(
            prev_gray, gray, prev_points, None, **LK_PARAMS
        )

        # Keep only successfully tracked points
        good_prev = prev_points[status == 1]
        good_next = next_points[status == 1]

        # Draw arrows
        for prev_pt, next_pt in zip(good_prev, good_next):

            x1, y1 = map(int, prev_pt.ravel())
            x2, y2 = map(int, next_pt.ravel())

            movement = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

            if movement > MIN_MOVEMENT:
                cv2.arrowedLine(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.circle(frame, (x1, y1), 3, (0, 0, 255), -1)

    # Refresh points every frame
    prev_points = cv2.goodFeaturesToTrack(
        gray,
        maxCorners=MAX_CORNERS,
        qualityLevel=QUALITY,
        minDistance=MIN_DISTANCE
    )

    prev_gray = gray

    cv2.imshow("Optical Flow", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

#### `cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)`
- optical flow works on grayscale
- color adds no useful information for movement detection
- reduces computation

#### `cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_points, None, **LK_PARAMS)`
- `prev_gray` — previous frame
- `gray` — current frame
- `prev_points` — points to track from previous frame
- `None` — output array, we let OpenCV allocate it
- returns `next_points`, `status`, `error`

#### `status == 1`
- status is 1 if point was tracked successfully
- status is 0 if point was lost (moved out of frame, occluded)
- we only keep successfully tracked points

#### `.ravel()`
- points are stored as shape `(1, 2)` by OpenCV
- `.ravel()` flattens to `(2,)` so we can unpack as `x, y`

#### `cv2.arrowedLine()`
- draws an arrow from previous position to new position
- direction shows movement direction
- length shows how far the point moved

#### `cv2.circle()`
- marks the original point position in red
- makes it easier to see where tracking started

#### Refreshing points every frame
- we recalculate `goodFeaturesToTrack` every frame
- ensures we always have fresh, high quality points
- without this, tracked points drift and quality degrades over time

### What To Expect

Still background:
- no arrows anywhere
- points detected but movement = 0

Moving hand across frame:
- arrows appear on hand
- direction matches movement direction
- background stays still

Moving camera:
- arrows appear everywhere
- all pointing same direction
- this is called global motion

### Summary

| Concept | What It Means |
| ------- | ------------- |
| Optical flow | pixel movement between two frames |
| Sparse | track specific points only, not every pixel |
| Dense | track every pixel, computationally heavy |
| `goodFeaturesToTrack` | finds corners reliable enough to track |
| Cornerness | uniqueness of a pixel — makes it trackable |
| `calcOpticalFlowPyrLK` | Lucas-Kanade sparse optical flow |
| `status == 1` | point was tracked successfully |
| `.ravel()` | flattens OpenCV point shape from `(1,2)` to `(2,)` |
| `arrowedLine` | draws movement direction and magnitude |

Next up — motion detection. Instead of tracking specific points, we detect regions of the frame where significant change has occurred.